# 02 — Tratamento de Dados
## LH Nautical | Bronze → Silver → Gold

**Objetivo:** Aplicar todas as transformações identificadas na EDA e construir
as camadas Silver (dados limpos) e Gold (dados analíticos com métricas financeiras).

```
Bronze (bruto)  ->  Silver (limpo/padronizado)  ->  Gold (modelo estrela + metricas)
```

**Princípio de imutabilidade:** Os dados Bronze nunca são modificados.
Cada camada é uma transformação independente e auditável.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.ingestion.ingest import load_all_raw_data
from src.processing.transform import (
    clean_vendas, clean_clientes, clean_produtos, clean_custos,
    build_silver_vendas, build_gold_fato_vendas, build_gold_dims,
    save_to_silver, save_to_gold,
)

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
print('Modulos carregados!')

Modulos carregados!


In [2]:
# Carrega dados da camada Bronze
raw = load_all_raw_data()

print('Camada Bronze carregada:')
for nome, df in raw.items():
    print(f'  {nome:10}: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

2026-05-11 23:04:19 | INFO | ingest               | =======================================================
2026-05-11 23:04:19 | INFO | ingest               | INGESTÃO — LH Nautical | Fonte: data/bronze/
2026-05-11 23:04:19 | INFO | ingest               | =======================================================
2026-05-11 23:04:19 | INFO | ingest               | [Bronze] vendas_2023_2024.csv → 9895 linhas × 6 colunas
2026-05-11 23:04:19 | INFO | ingest               | [Bronze] produtos_raw.csv → 157 linhas × 4 colunas
2026-05-11 23:04:19 | INFO | ingest               | [Bronze] clientes_crm.json → 49 linhas × 4 colunas
2026-05-11 23:04:19 | INFO | ingest               | [Bronze] custos_importacao.json → 150 linhas × 4 colunas
2026-05-11 23:04:19 | INFO | ingest               | Carregados: ['vendas', 'produtos', 'clientes', 'custos'] | INGESTÃO CONCLUÍDA


Camada Bronze carregada:
  vendas    : 9,895 linhas x 6 colunas
  produtos  : 157 linhas x 4 colunas
  clientes  : 49 linhas x 4 colunas
  custos    : 150 linhas x 4 colunas


---
## 1. Bronze → Silver: Limpeza e Padronização

### 1.1 Vendas

In [3]:
silver_vendas = clean_vendas(raw['vendas'])

print('=== Vendas: Bronze -> Silver ===')
print(f'Linhas Bronze : {len(raw["vendas"]):,}')
print(f'Linhas Silver : {len(silver_vendas):,}')
print(f'\nColunas Silver: {list(silver_vendas.columns)}')
print()

# Verificacao: datas parseadas corretamente
print('data_venda (primeiros 5):', silver_vendas['data_venda'].head().dt.strftime('%Y-%m-%d').tolist())
print('Nulos em data_venda:', silver_vendas['data_venda'].isnull().sum())

# Verificacao: colunas temporais
print('\nColunas temporais criadas:')
display(silver_vendas[['data_venda','ano','mes','trimestre','dia_semana']].head(5))

2026-05-11 23:04:19 | INFO | transform            | [Silver] Vendas: 9895 linhas


=== Vendas: Bronze -> Silver ===
Linhas Bronze : 9,895
Linhas Silver : 9,895

Colunas Silver: ['id_venda', 'id_cliente', 'id_produto', 'quantidade', 'receita_bruta', 'data_venda', 'status_venda', 'desconto_pct', 'ano', 'mes', 'trimestre', 'dia_semana']

data_venda (primeiros 5): ['2023-09-10', '2024-09-15', '2024-08-13', '2023-02-03', '2024-02-12']
Nulos em data_venda: 0

Colunas temporais criadas:


,data_venda,ano,mes,trimestre,dia_semana
0,2023-09-10,2023,9,3,Sunday
1,2024-09-15,2024,9,3,Sunday
2,2024-08-13,2024,8,3,Tuesday
3,2023-02-03,2023,2,1,Friday
4,2024-02-12,2024,2,1,Monday


### 1.2 Produtos

In [4]:
silver_produtos = clean_produtos(raw['produtos'])

print('=== Produtos: Bronze -> Silver ===')
print(f'Linhas Bronze (com dups) : {len(raw["produtos"]):,}')
print(f'Linhas Silver (sem dups) : {len(silver_produtos):,}')
print(f'Duplicatas removidas     : {len(raw["produtos"]) - len(silver_produtos)}')
print()

# Verificacao: categorias normalizadas
print('Categorias Bronze (variacoes):', raw['produtos']['actual_category'].nunique(), 'distintas')
print('Categorias Silver (canonicas):', silver_produtos['categoria'].unique().tolist())

# Verificacao: preco como float
print(f'\nTipo de preco_venda: {silver_produtos["preco_venda"].dtype}')
print('Amostra preco_venda:', silver_produtos['preco_venda'].head(3).tolist())

2026-05-11 23:04:19 | WARNING | transform            | [Produtos] 7 linha(s) duplicada(s) removida(s) (mesmo id_produto)
2026-05-11 23:04:19 | INFO | transform            | [Silver] Produtos: 150 linhas | Categorias: ['Eletrônicos', 'Propulsão', 'Ancoragem']


=== Produtos: Bronze -> Silver ===
Linhas Bronze (com dups) : 157
Linhas Silver (sem dups) : 150
Duplicatas removidas     : 7

Categorias Bronze (variacoes): 39 distintas
Categorias Silver (canonicas): ['Eletrônicos', 'Propulsão', 'Ancoragem']

Tipo de preco_venda: float64
Amostra preco_venda: [33122.52, 13998.15, 9024.19]


### 1.3 Clientes

In [5]:
silver_clientes = clean_clientes(raw['clientes'])

print('=== Clientes: Bronze -> Silver ===')
print(f'Colunas Silver: {list(silver_clientes.columns)}')
print()

# Verificacao: location parseado para cidade/estado
print('location (Bronze) -> cidade + estado (Silver):')
display(silver_clientes[['localizacao_raw','cidade','estado']].head(8))

# E-mails flagados
n_inv = (~silver_clientes['email_valido']).sum()
print(f'\nE-mails invalidos flagados: {n_inv} ({n_inv/len(silver_clientes)*100:.0f}%)')

2026-05-11 23:04:19 | WARNING | transform            | [Clientes] 30 e-mail(s) inválido(s)
2026-05-11 23:04:19 | INFO | transform            | [Silver] Clientes: 49 linhas


=== Clientes: Bronze -> Silver ===
Colunas Silver: ['nome_completo', 'localizacao_raw', 'id_cliente', 'email', 'cidade', 'estado', 'email_valido']

location (Bronze) -> cidade + estado (Silver):


,localizacao_raw,cidade,estado
0,"Aratu (Candeias) , BA",Aratu (Candeias),BA
1,"PE , Recife",Recife,PE
2,"Rio Grande,RS",Rio Grande,RS
3,"AC , Rio Branco",Rio Branco,AC
4,PA - Santarém Novo,Santarém Novo,PA
5,"Fortaleza do Tabocão , TO",Fortaleza do Tabocão,TO
6,PB/Cabedelo,Cabedelo,PB
7,SE - Aracaju,Aracaju,SE



E-mails invalidos flagados: 30 (61%)


### 1.4 Custos

In [6]:
silver_custos = clean_custos(raw['custos'])

print('=== Custos: Bronze -> Silver ===')
print(f'Colunas Silver: {list(silver_custos.columns)}')
print()

# Verificacao: custo convertido para BRL
print('Taxa de cambio aplicada: R$ 5.10/USD')
display(silver_custos.head(5))

2026-05-11 23:04:20 | INFO | transform            | [Silver] Custos: 150 linhas (taxa câmbio: R$ 5.10/USD)


=== Custos: Bronze -> Silver ===
Colunas Silver: ['id_produto', 'custo_unitario_usd', 'custo_unitario_brl']

Taxa de cambio aplicada: R$ 5.10/USD


,id_produto,custo_unitario_usd,custo_unitario_brl
0,1,5579.75,28456.72
1,2,2602.27,13271.58
2,3,1753.77,8944.23
3,4,583.85,2977.64
4,5,4285.30,21855.03


In [7]:
# Salva camada Silver
for nome, df in [('vendas', silver_vendas), ('clientes', silver_clientes),
                 ('produtos', silver_produtos), ('custos', silver_custos)]:
    save_to_silver(df, nome)

print('[OK] Camada Silver salva em data/silver/')

2026-05-11 23:04:20 | INFO | transform            | [Silver] Salvo: vendas.parquet (9895 linhas)
2026-05-11 23:04:20 | INFO | transform            | [Silver] Salvo: clientes.parquet (49 linhas)
2026-05-11 23:04:20 | INFO | transform            | [Silver] Salvo: produtos.parquet (150 linhas)
2026-05-11 23:04:20 | INFO | transform            | [Silver] Salvo: custos.parquet (150 linhas)


[OK] Camada Silver salva em data/silver/


---
## 2. Silver → Gold: Enriquecimento Financeiro

Nesta etapa, criamos as **métricas financeiras** que não existem nos dados brutos:
- `canal_venda` derivado do estado do cliente (SC = loja_fisica, demais = ecommerce)
- `custo_total`, `lucro_bruto`, `margem_pct` calculados a partir do custo de importação

In [8]:
# Enriquece vendas com dados de produto, cliente e custo
silver_enriched = build_silver_vendas(silver_vendas, silver_produtos, silver_clientes, silver_custos)

print(f'Silver enriquecido: {silver_enriched.shape[0]:,} linhas, {silver_enriched.shape[1]} colunas')

cols_fin = ['receita_bruta', 'desconto_pct', 'desconto_valor', 'receita_liquida',
            'custo_total', 'lucro_bruto', 'margem_pct', 'canal_venda']

print('\nMetricas financeiras (describe):')
display(silver_enriched[[c for c in cols_fin if c in silver_enriched.columns]].describe().round(2))

2026-05-11 23:04:20 | WARNING | transform            | [Silver+] 2424 transações com margem negativa (revisar custo)
2026-05-11 23:04:20 | INFO | transform            | [Silver+] Vendas enriquecidas: 9895 linhas, 24 colunas


Silver enriquecido: 9,895 linhas, 24 colunas

Metricas financeiras (describe):


,receita_bruta,desconto_pct,desconto_valor,receita_liquida,custo_total,lucro_bruto,margem_pct
count,9895.00,9895.0,9895.0,9895.00,9895.00,9895.00,9895.00
mean,263797.83,0.0,0.0,263797.83,253731.91,10065.92,3.65
std,390007.18,0.0,0.0,390007.18,376925.85,30869.11,5.98
min,294.50,0.0,0.0,294.50,272.19,-194764.95,-13.19
25%,23138.20,0.0,0.0,23138.20,22247.54,33.79,0.20
50%,82225.00,0.0,0.0,82225.00,78839.88,1657.60,4.14
75%,339094.50,0.0,0.0,339094.50,326103.93,10744.59,7.51
max,2222973.00,0.0,0.0,2222973.00,2146621.35,252473.65,17.31


In [9]:
# Alerta: transacoes com margem negativa
margem_neg = silver_enriched[silver_enriched['margem_pct'] < 0]
print(f'Transacoes com margem negativa: {len(margem_neg):,} ({len(margem_neg)/len(silver_enriched)*100:.1f}%)')
print(f'Receita total negativa-margem  : R$ {margem_neg["receita_liquida"].sum():,.2f}')
print(f'Lucro total negativa-margem    : R$ {margem_neg["lucro_bruto"].sum():,.2f}')
print('\n[!] 30 produtos vendidos sistematicamente abaixo do custo de importacao')
print('    Detalhe na analise 03_analise_vendas.ipynb')

Transacoes com margem negativa: 2,424 (24.5%)
Receita total negativa-margem  : R$ 646,730,374.05
Lucro total negativa-margem    : R$ -25,479,112.09

[!] 30 produtos vendidos sistematicamente abaixo do custo de importacao
    Detalhe na analise 03_analise_vendas.ipynb


In [10]:
# Constroi tabelas Gold (modelo estrela)
fato_vendas = build_gold_fato_vendas(silver_enriched)
dims        = build_gold_dims(silver_clientes, silver_produtos, silver_custos)

# Salva
save_to_gold(fato_vendas, 'fato_vendas')
for nome, df in dims.items():
    save_to_gold(df, nome)

print('[OK] Camada Gold salva em data/gold/')
print(f'  fato_vendas   : {len(fato_vendas):,} linhas, {fato_vendas.shape[1]} colunas')
for nome, df in dims.items():
    print(f'  {nome:15}: {len(df):,} linhas')

2026-05-11 23:04:20 | INFO | transform            | [Gold] fato_vendas: 9895 linhas × 18 colunas
2026-05-11 23:04:20 | INFO | transform            | [Gold] dim_clientes: 49 linhas
2026-05-11 23:04:20 | INFO | transform            | [Gold] dim_produtos: 150 linhas
2026-05-11 23:04:20 | INFO | transform            | [Gold] Salvo: fato_vendas.parquet (9895 linhas)
2026-05-11 23:04:20 | INFO | transform            | [Gold] Salvo: dim_clientes.parquet (49 linhas)
2026-05-11 23:04:20 | INFO | transform            | [Gold] Salvo: dim_produtos.parquet (150 linhas)


[OK] Camada Gold salva em data/gold/
  fato_vendas   : 9,895 linhas, 18 colunas
  dim_clientes   : 49 linhas
  dim_produtos   : 150 linhas


---
## 3. Validação da Camada Gold

In [11]:
print('=== VALIDACAO DA CAMADA GOLD ===')

# 1. Sem nulos em colunas criticas
criticas = ['id_venda', 'id_cliente', 'id_produto', 'receita_liquida', 'lucro_bruto']
nulos = fato_vendas[criticas].isnull().sum()
print('\nNulos em colunas criticas (esperado: 0):')
print(nulos)

# 2. Contagens integras
print(f'\nTotal transacoes       : {len(fato_vendas):,}')
print(f'Clientes distintos     : {fato_vendas["id_cliente"].nunique():,}')
print(f'Produtos distintos     : {fato_vendas["id_produto"].nunique():,}')
print(f'Periodo coberto        : {fato_vendas["data_venda"].min().date()} a {fato_vendas["data_venda"].max().date()}')

# 3. Sanidade financeira
print(f'\nReceita liquida total  : R$ {fato_vendas["receita_liquida"].sum():>18,.2f}')
print(f'Lucro bruto total      : R$ {fato_vendas["lucro_bruto"].sum():>18,.2f}')
print(f'Margem media           :    {fato_vendas["margem_pct"].mean():>16.1f}%')

print('\n[OK] Validacao concluida - Gold pronto para analise')

=== VALIDACAO DA CAMADA GOLD ===

Nulos em colunas criticas (esperado: 0):
id_venda           0
id_cliente         0
id_produto         0
receita_liquida    0
lucro_bruto        0
dtype: int64

Total transacoes       : 9,895
Clientes distintos     : 49
Produtos distintos     : 150
Periodo coberto        : 2023-01-01 a 2024-12-31

Receita liquida total  : R$   2,610,279,510.70
Lucro bruto total      : R$      99,602,307.41
Margem media           :                 3.7%

[OK] Validacao concluida - Gold pronto para analise


---
## 4. Resumo da Transformação

| Camada | Arquivo              | Linhas | Colunas | Descrição                                  |
|--------|----------------------|--------|---------|--------------------------------------------|
| Bronze | vendas_2023_2024.csv | 9.895  | 6       | Dados brutos: id, id_client, qtd, total, sale_date |
| Bronze | produtos_raw.csv     | 157    | 4       | name, price (string), code, actual_category |
| Bronze | clientes_crm.json    | 49     | 4       | full_name, location, email, code           |
| Bronze | custos_importacao.json | 150  | 4       | product_id, product_name, historic_data    |
| Silver | vendas.parquet       | 9.895  | 12      | Datas parseadas, colunas temporais         |
| Silver | produtos.parquet     | 150    | 4       | Preço numérico, categoria canônica, sem dup |
| Silver | clientes.parquet     | 49     | 7       | cidade + estado extraídos, email_valido    |
| Silver | custos.parquet       | 150    | 3       | Último preço USD convertido para BRL       |
| Gold   | fato_vendas.parquet  | 9.895  | 18      | Métricas financeiras + canal_venda         |
| Gold   | dim_clientes.parquet | 49     | 6       | Dimensão clientes                          |
| Gold   | dim_produtos.parquet | 150    | 6       | Dimensão produtos                          |

> **Próximo passo:** `03_analise_vendas.ipynb` — análise de negócio sobre a camada Gold.